# Redes neuronales

Dentro del deep learning, existen dos maneras de crear una red neuronal, desde cero o, lo m?s com?n, el mediante el m?todo de *transfer learning*, este consiste en coger una red ya preentrenada para otra tarea y adaptarla a tu problema, m?s adelante lo explicaremos con m?s detalle

Vamos a ver c?mo hacer esto con Pytorch

## Construyendo una red neuronal desde cero

En la librer?a [torch.nn](https://pytorch.org/docs/stable/nn.html) tenemos todo lo necesario para construir redes neuronales.

Para crear redes neuronales desde cero lo podemos hacer de muchas maneras, pero lo mejor es hacerlas modulares, para as? poder escalar la red si hace falta, para ello, tenemos que crear una clase que herede de [torch.nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html). Adem?s tenemos que crear los m?todos ``__init__`` y ``forward``.

 * Con el m?todo ``__init__`` llamaremos al constructor de la clase de la que se hereda (``torch.nn.Module``) y adem?s inicializaremos nuestra red con las capas que queramos que tenga
 * Con el m?todo ``forward`` Pytorch nos permite llamar a la red como una funci?n, aunque la hemos definido como una clase. Ahora veremos esto

Creamos una red para que aprenda a clasificar las im?genes del conjunto de datos ``CIFAR-10`` que hemos visto antes. Por recordar, ``CIFAR-10`` es un conjunto de datos de im?genes que consta de 50.000 ejemplos de entrenamiento y 10.000 ejemplos de test. Cada ejemplo contiene una imagen a color de 3x32x32 y una etiqueta asociada de una de las 10 posibles clases.

Como vamos a tener im?genes de 32x32 p?xeles a color esto significa que vamos a tener matrices de 3x32x32.

De modo que lo primero que vamos a hacer es *aplanar* estas matrices, es decir, vamos a pasar estas matrices de 3x32x32 a un vector de 3072 valores (3x32x32 = 3072)

Aqu? podemos ver un ejemplo de c?mo se aplana una matriz de 4x4

<div style="text-align:center;">
  <img src="Animaciones/Aplanar_matriz.gif" alt="Aplanar"> <!-- style="width:425px;height:626px;"> -->
</div>

Una vez tenemos la imagen aplanada podemos pasarla por una red MLP (multi layer perceptron). En este caso haremos un m?dulo que contiene una red de dos capas

In [ ]:
from torch import nn

class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()   # Se inicializa el m?dulo nn.Module
        self.flatten = nn.Flatten(start_dim=0)  # Se crea una primera capa que aplana la imagen de entrada
        self.linear_relu_stack = nn.Sequential( # Se crea una m?dulo de arquitectura secuencial:
            nn.Linear(3*32*32, 512),                # Se a?ade una primera capa lineal que est? preparada 
                                                    # para que le entre un vector de 3x32x32 (3072)
                                                    # y sacar? un vector de 512
            nn.ReLU(),                              # Se a?ade una no linealidad
            nn.Linear(512, 512),                    # Se a?ade una segunda capa lineal que le entran 512 
                                                    # datos y saca 512 datos
            nn.ReLU(),                              # Se a?ade una no linealidad
            nn.Linear(512, 10)                      # Se a?ade una tercera capa lineal que le entran 512 
                                                    # datos y saca un array de tama?o 10 (el n?mero
                                                    # de etiquetas)
        )
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.flatten(x)                         # Se pasa la imagen por la capa de aplanado
        x.unsqueeze_(0)                             # Se a?ade una dimensi?n para que todo cuadre
        logits = self.linear_relu_stack(x)          # Se pasa el vector resultante por la red
        probs = self.softmax(logits)                # Se pasa el resultado por la funci?n softmax
        return probs

Hemos definido la red neuronal, es decir, le hemos dicho a Pytorch c?mo queremos que sea, ahora hace falta instanciarla, es decir, construirla

In [ ]:
model = NeuralNetwork()

Una vez tenemos la red neuronal creada podemos imprimir su estructura

In [ ]:
print(model)

Vamos a probar que nuestra red funciona, creamos una matriz de n?meros aleatorios de 3x32x32 y se lo pasamos a la 
red

In [ ]:
import torch

input = torch.rand(3, 32, 32)
output = model(input)

output

La red devuelve un vector con las probabilidades de cada clase. La posici?n donde se encuentra el m?ximo de estos valores corresponde a la clase que la red cree que pertenece la matriz que le hemos metido

In [ ]:
posicion = output.argmax(1)

posicion.item()

Ya sabemos la posici?n, veamos por casualidad qu? valor tiene la salida en esa posici?n

In [ ]:
output[posicion]

Obtenemos un error, esto es porque la salida no es un vector, sino una matriz de tama?o 1x10

In [ ]:
output.shape

As? que usamos el m?todo ``squeeze()`` que ya vimos en la clase anterior para quitar la dimensi?n que tiene valor 1

In [ ]:
output.squeeze().shape, output.squeeze()[posicion].item()

Ya vemos que con el m?todo ``squeeze()`` tenemos un vector, y que podemos ver la salida en la posici?n con el valor m?ximo

Hacemos un diccionario con todas las posibles clases para ver qu? ha predicho la red

In [ ]:
labels_map = {
    0: "airplane",
    1: "automobile",
    2: "bird",
    3: "cat",
    4: "deer",
    5: "dog",
    6: "frog",
    7: "horse",
    8: "ship",
    9: "truck",
}

labels_map[posicion.item()]

### Par?metros del modelo

Puede que nos interese ver los par?metros de la red neuronal. En este ejemplo iteramos por todas las capas e imprimimos su tama?o y sus valores

In [ ]:
print("Model structure: ", model, "\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

## Construyendo una red neuronal mediante *transfer learning*

Lo anterior est? muy bien, podemos crearnos una red neuronal a nuestro gusto, con la arquitectura que queramos. Esto es una de las cualidades de Pytorch, que podemos construir muy facilmente cualquier estructura de red. 
En el ejemplo anterior hemos construido una red neuronal secuencial, es decir la salida de capa entrada era la entrada de la siguiente capa, pero muy facilmente en el m?todo ``forward`` podiamos haber hecho las conexiones que nos hubiera dado la gana

Pero hacer una red neuronal desde cero tiene el problema de que los par?metros se crean aleatoriamente, por lo que hay que entrenarla desde cero.

Por ello, en la mayor?a de los casos lo que se hace es aprovechar el *conocimiento* de otras redes neuronales. Redes entrenadas para otros prop?sitos, pero que nos pueden valer.

Existe una competici?n llamada ``ImageNet``, en la cual se tiene que conseguir clasificar im?genes de una base de datos con m?s de 14 millones de im?genes, las cuales pertenecen a 1000 categor?as distintas.

A esta competici?n se presentan grupos de investigaci?n de todo el mundo para poner a prueba las redes neuronales que han desarrollado. Por suerte estas redes ya entrenadas para esta competici?n est?n disponibles para todo el mundo.
Por lo que nosotros podemos coger una de estas redes y adaptarlas para nuestro problema

Por ejemplo, una de estas redes preentrenadas es la ``Resnet18``. Hay varios tipos de Resnet, el n?mero es la cantidad de capas, por lo que en nuestro caso solo tien 18 capas. Hay ``Resnet34``, con 34 capas; ``Resnet50``, con 50 capas y varias m?s hasta la ``Resnet152``, con 152 capas.

Nos quedamos con las ``Resnet18``, porque es un modelo peque?o y para lo que queremos hacer ahora nos vale.

Pytorch tiene todas estas `Resnet` y otras muchas m?s [redes de visi?n](https://pytorch.org/vision/stable/models.html) disponibles para descargar

 > Importante: Para descargarse la red preentrenada hay que poner el par?metro `pretrained` a True

In [ ]:
from torchvision import models

model = models.resnet18(pretrained=True)

model

Al imprimir la red podemos ver todas las capas que tiene, pero en especial podemos ver una capa llamada ``fc`` que es una capa lineal que tiene 512 caracter?sticas de entrada y 1000 caracter?sticas de salida.

```(fc): Linear(in_features=512, out_features=1000, bias=True)```

No es el objetivo de esta clase explicar la ``Resnet18``, por lo que solo diremos que a groso modo, ``Resnet18`` tiene un mont?n de capas entrenadas para el conjunto de datos de ``ImageNet``, por eso, en la ?ltima capa hay 1000 caracter?sticas de salida, correspondientes a las 1000 clases de ``ImageNet``.

En la parte de visi?n se explicar? en detalle este tipo de redes, pero a groso modo, las primeras capas de la red son capaces de reconocer formas simples como lineas o esquinas. Y a medida que vamos adentr?ndonos en las capas, la red es capad de reconocer formas m?s complejas, como ojos, caras, etc. Hasta llegar a la ?ltima capa donde es capaz de reconocer los objetos de ``ImageNet``, como coches, aviones, perros, ...

Lo que vamos a hacer nosotros es eliminar la ?ltima capa, la llamada ``fc``, que tiene 1000 neuronas, para las 1000 clases de ``ImageNet`` y la vamos a sustituir por otra capa lineal, pero en este caso de 10 neuronas, para las 10 clases de nuestro problema

In [ ]:
model.fc = nn.Linear(512, 10)

Si ahora imprimimos el modelo veremos que la ?ltima capa ha cambiado y ahora solo tiene 10 neuronas a la salida

In [ ]:
model

Vamos a probar que nuestra red funciona, creamos una matriz de n?meros aleatorios de 64x3x32x32 y se lo pasamos a la red. Ahora se a?ade la dimensi?n inicial con valor 64, porque la red espera un tensor en el que el primer elemento es el numero de baches ,luego el n?mero de canales y por ?ltimo el alto y el ancho. De modo que hacemos como que tenemos un bach de 64 im?genes

In [ ]:
import torch

input = torch.rand(64, 3, 32, 32)
output = model(input)

output[0]

La red devuelve un vector con los valores de cada clase.

Para obtener las probabilidades de cada clase, a la salida le tenemos que pasar por la funci?n `softmax`

La posici?n donde se encuentra la probabilidad m?xima de estos valores corresponde a la clase que la red cree que pertenece la matriz que le hemos metido

In [ ]:
probabilidades = output.softmax(dim=1)

posiciones = probabilidades.argmax(1)

posicion = posiciones[0].item()

posicion

Ya sabemos la posici?n, veamos por casualidad qu? valor tiene la salida en esa posici?n

In [ ]:
output[0][posicion]

Hacemos un diccionario con todas las posibles clases para ver qu? ha predicho la red

In [ ]:
labels_map = {
    0: "airplane",
    1: "automobile",
    2: "bird",
    3: "cat",
    4: "deer",
    5: "dog",
    6: "frog",
    7: "horse",
    8: "ship",
    9: "truck",
}

labels_map[posicion]